## 패키지 선언

In [1]:
import torch
import torch.nn as nn
import numpy as np
import torchvision.datasets as dataset
import torchvision.transforms as transform
from torch.utils.data import DataLoader
from torchsummary import summary

## Dataset 선언

In [2]:
# Training dataset 다운로드
cifar100_train = dataset.CIFAR100(root = "./", # 데이터셋을 저장할 위치
                            train = True,
                            transform = transform.ToTensor(),
                            download = True)
# Testing dataset 다운로드
cifar100_test = dataset.CIFAR100(root = "./",
                            train = False,
                            transform = transform.ToTensor(),
                            download = True)

100%|██████████| 169M/169M [00:06<00:00, 24.4MB/s]


## 문제 1. 모델 정의

In [3]:
class modelOne(nn.Module):
  def __init__(self):
    super(modelOne, self).__init__()

    self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride = 1, padding=2)
    self.conv1_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride = 1, padding=2)

    self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, stride = 1, padding=2)
    self.conv2_2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride = 1, padding=2)

    self.conv3_1 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride = 1, padding=1)
    self.conv3_2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride = 1, padding=1)

    self.fc1 = nn.Linear(4096, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 100)

    # 파라미터를 가지지 않은 layer는 한 번만 선언해도 문제 없음
    self.relu = nn.ReLU()
    self.AvgPool2d = nn.AvgPool2d(kernel_size=2, stride=2)

  def forward(self, x):

    out = self.relu(self.conv1_1(x))
    out = self.relu(self.conv1_2(out))
    out = self.AvgPool2d(out)

    out = self.relu(self.conv2_1(out))
    out = self.relu(self.conv2_2(out))
    out = self.AvgPool2d(out)

    out = self.relu(self.conv3_1(out))
    out = self.relu(self.conv3_2(out))
    out = self.AvgPool2d(out)

    # 평탄화
    out = out.reshape(-1, 4096)

    # fully connected layers
    out = self.relu(self.fc1(out))
    out = self.relu(self.fc2(out))
    out = self.fc3(out)

    return out

## Parameter 확인

In [4]:
from torchsummary import summary
network = modelOne().to('cpu')
summary(network, (3, 32, 32), device='cpu')
print(summary)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]           1,216
              ReLU-2           [-1, 16, 32, 32]               0
            Conv2d-3           [-1, 32, 32, 32]          12,832
              ReLU-4           [-1, 32, 32, 32]               0
         AvgPool2d-5           [-1, 32, 16, 16]               0
            Conv2d-6           [-1, 32, 16, 16]          25,632
              ReLU-7           [-1, 32, 16, 16]               0
            Conv2d-8           [-1, 64, 16, 16]          51,264
              ReLU-9           [-1, 64, 16, 16]               0
        AvgPool2d-10             [-1, 64, 8, 8]               0
           Conv2d-11            [-1, 128, 8, 8]          73,856
             ReLU-12            [-1, 128, 8, 8]               0
           Conv2d-13            [-1, 256, 8, 8]         295,168
             ReLU-14            [-1, 25

## 문제 2. 모델 정의

In [5]:
class modelTwo(nn.Module):
  def __init__(self):
    super(modelTwo, self).__init__()

    self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, stride = 1, padding=3)

    self.conv2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=5, stride = 1, padding=2)

    self.conv3 = nn.Conv2d(in_channels=64, out_channels=256, kernel_size=3, stride = 1, padding=1)

    self.resdual1_1 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
    self.resdual1_2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)

    self.resdual1_3 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
    self.resdual1_4 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)

    self.resdual1_5 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
    self.resdual1_6 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)

    self.resdual2_1 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
    self.resdual2_2 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)

    self.resdual2_3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
    self.resdual2_4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)

    # 파라미터를 가지지 않은 layer는 한 번만 선언해도 문제 없음

    self.avgPool2d_1 = nn.AvgPool2d(kernel_size=2, stride=2)
    self.avgPool2d_2 = nn.AvgPool2d(kernel_size=4, stride=2, padding = 1)
    self.maxPool2d = nn.MaxPool2d(kernel_size=2, stride=2)

    # Channel Attention 을 위한 convolution layer 선언
    self.GAP = nn.AdaptiveAvgPool2d((1, 1)) #들어오는 크기만큼 풀링을 해주겠다
    self.conv_CA1= nn.Conv2d(in_channels = 64, out_channels = 32, kernel_size = 1, stride = 1, padding = 0)
    self.conv_CA2= nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 1, stride = 1, padding = 0)


    # Spatial Attention 을 위한 convolution layer 선언
    self.conv_SA1 = nn.Conv2d(in_channels=64, out_channels=1, kernel_size=1, stride = 1, padding=0)
    self.conv_SA2 = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=5, stride = 1, padding=2)
    self.sigmoid = nn.Sigmoid()
    self.relu = nn.ReLU()

    self.fc1 = nn.Linear(4096, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 100)

  def forward(self, x):

    out = self.relu(self.conv1(x))
    skip1 = self.avgPool2d_1(out)

    out = self.relu(self.resdual1_1(skip1))
    out = self.resdual1_2(out)
    skip2 = skip1 + out
    out = self.relu(self.resdual1_3(skip2))
    out = self.resdual1_4(out)
    skip3 = skip2 + out
    out = self.relu(self.resdual1_5(skip3))
    out = self.resdual1_6(out)
    out = skip3 + out

    out = skip1 + out
    out = torch.cat([out, skip1], dim = 1)

    out = self.relu(self.conv2(out))
    out = self.avgPool2d_2(out)

    weight = self.GAP(out)
    weight = self.conv_CA1(weight)
    weight = self.relu(weight)
    weight = self.conv_CA2(weight)
    weight = self.sigmoid(weight)
    out = out * weight

    weight2 = self.conv_SA1(out)
    weight2 = self.conv_SA2(weight2)
    weight2 = self.sigmoid(weight2)
    skip4 = out * weight2

    out = self.relu(self.resdual2_1(skip4))
    out = self.resdual2_2(out)
    skip5 = skip4 + out
    out = self.relu(self.resdual2_3(skip5))
    out = self.resdual2_4(out)
    out = skip5 + out

    out = self.relu(self.conv3(out))
    out = self.maxPool2d(out)

    # 평탄화
    out = out.reshape(-1, 4096)

    # fully connected layers
    out = self.relu(self.fc1(out))
    out = self.relu(self.fc2(out))
    out = self.fc3(out)
    return out

## Parameter 확인

In [ ]:
from torchsummary import summary
network = modelTwo().to('cpu')
summary(network, (3, 32, 32), device='cpu')
print(summary)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 32, 32, 32]           4,736
              ReLU-2           [-1, 32, 32, 32]               0
         AvgPool2d-3           [-1, 32, 16, 16]               0
            Conv2d-4           [-1, 32, 16, 16]           9,248
              ReLU-5           [-1, 32, 16, 16]               0
            Conv2d-6           [-1, 32, 16, 16]           9,248
            Conv2d-7           [-1, 32, 16, 16]           9,248
              ReLU-8           [-1, 32, 16, 16]               0
            Conv2d-9           [-1, 32, 16, 16]           9,248
           Conv2d-10           [-1, 32, 16, 16]           9,248
             ReLU-11           [-1, 32, 16, 16]               0
           Conv2d-12           [-1, 32, 16, 16]           9,248
           Conv2d-13           [-1, 64, 16, 16]         102,464
             ReLU-14           [-1, 64,